### GPT 1

In [ ]:
# dependencies 
import math 
import torch
import torch.nn as nn 
import torch.nn.functional as F
from pydantic import BaseModel, Field, model_validator

In [ ]:
# Configuration
class GPT1Config(BaseModel):
    vocab_size: int = Field(default = 40478, gt = 0, description = "Vocabulary size")

    cotext_length: int = Field(default = 572, gt = 0, description = "Max cotext length/Block size")
    hidden_size: int = Field(default = 768, gt = 0, description = " model/hidden dimension aka d_model") # embed_dim == hidden_size == n_embed == d_model
    n_layer: int = Field(default = 12, gt = 0, description = "number of stacked decoder blocks")
    num_head: int = Field(default = 12, gt = 0, description = "number of attention heads")
    intermediate_step: int = Field(default = 3072, gt = 0, description = "FFN inner dimension")
    dropout: float = Field(default = 0.1, ge = 0.0, description = "Residual dropout")
    attention_dropout: float = Field(default = 0.1, ge = 0.0, description = "attention dropout")
    embedding_dropout: float = Field(default = 0.1, ge = 0.0, description = "embedding dropout")

    model_config = {"freeze": True} 

    @model_validator(mode = "after") # operates over the entire model input, whereas the field_validator only operates on an isolated class where it define
    def check_head_divide_embed(self):
        if self.n_embed % self.num_head != 0:
            raise ValueError(
                f"n_embed ({self.n_embed} must be divisable by num_heads ({self.num_head});"
                f"got remainder {self.n_embed % self.num_head}"
                )
        return self 

In [ ]:
# Masked MultiHeadSelfAttention(nn.Module)
class MaskedMultiHeadSelfAttention(nn.Module):
    """
            Shapes (B = batch, T = sequence length, C = hidden_size, H = n_head, D = C/H):
            input  x        : (B, T, C)
            qkv projection   -> (B, T, 3C)  -> split into q,k,v : (B, T, C) each
            reshape to heads : (B, H, T, D)
            attention scores : (B, H, T, T)
            context          : (B, H, T, D) -> merge heads -> (B, T, C)
            output projection: (B, T, C)
        """
    def __init__(self, config: GPT1Config):
        super().__init__()
        self.num_head = config.num_head
        self.head_dim = config.hidden_size/config.num_head

        # only one matmul can produce Q, K, V together (faster than separate 3)
        self.qkv_proj = nn.Linear(config.hidden_size, 3 * config.hidden_size)
        self.out_ptoj = nn.Linear(config.hidden_size, config.hidden_size)

        self.attention_dropout = nn.Dropout(config.attention_dropout)
        self.residual_dropout = nn.Dropout(config.dropout)

        # perform the lower trianglular part
        causal_mask = torch.tril(torch.ones(config.context_len, config.context_length))
        
        self.register_buffer("causal_mask", causal_mask.view(1, 1, config.context_length, config.context_length))

    def forward(self, x):
        B, T, C = x.shape # [batch_size, seq_len, hidden_size]

        qkv = self.qkv_proj(x) # [B, T, 3C]
        q, k, v = qkv.split(C, dim = 2) # [B, T, C]

        # [B, H, T, D]
        q = q.view(B, T, self.num_head, self.head_dim).tranpose(1, 2)
        k = k.view(B, T, self.num_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_head, self.head_dim).tranpose(1,2)

        # scaled dot-product attention
        attention = q @ k.transpose(-2, -1) # [B, H, T, D] @ [B, H, D, T] --> [B, H, T, T]
        attention = attention / math.sqrt(self.head_dim) # scaling
        attention = attention.masked_fill(self.causal_mask[:, :, :T, :T] == 0, float("-inf"))
        attention = F.softmax(attention, dim = -1)
        attention = self.attention_dropout(attention)

        score = attention @ v
        score = score.transpose(1,2).contiguous().view(B, T, C) # merge the head (num_head * )

        return self.residual_dropout(self.out_proj(score))

In [ ]:
# Position-wise FFN 
class PositionWiseFNN(nn.Module):
    # (B,T,C) -> Linear: (B,T,d_ff) -> Activation: GELU -> Linear: (B,T,C)
    def __init__(self, config: GPT1Config):
        super().__init__()
        self.fc1 = nn.Linear(config.hidden_size, config.intermediate_step)
        self.fc2 = nn.Linear(config.intermediate_step, config.hidden_size)
        self.dropout = nn.DropOut(config.dropout)
        # nn.Linear only cares about the last dim of the tensor

    def forward(self):
        x = self.fc1(x)
        x = F.GELU(x)
        x = self.fc2(x)
        return self.dropout(x)

In [ ]:
# 1 Decoder Block 
class TransformerBlock(nn.Module):
    def __init__(self, config: GPT1Config):
        super().__init__()
        self.attention = MaskedMultiHeadSelfAttention(config)
        self.ln1 = nn.LayerNorm(config.hidden_size)
        self.ffn = PositionWiseFNN(config)
        self.ln2 = nn.LayerNorm(config.hidden_size)

    # Post-norm technique
    def forward(self, x):
        x = self.ln1(x + self.attention(x))
        x = self.ln2(x + self.ffn(x))
        return x

In [ ]:
class GPT1(nn.Module):
    def __init__(self, config: GPT1Config):
        super().__init__()
        self.config = config

        self.token_emb = nn.Embedding(config.vocab_size, config.hidden_size)
        self.position_emb = nn.Parameter(torch.zeros(config.context_length, config.hidden_size))
        self.embedding_dropout = nn.Dropout(config.dropout)

        self.blocks = nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layer)])

        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias = False)

        self.lm_head.weight = self.token_emb.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean = 0.0, std = 0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)

        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean = 0.0, std = 0.02)


    def forward(self, idx, targets = None):
        # idx : (B, T) LongTensor of token ids
        # targets : (B, T) LongTensor of next-token ids
        B, T = idx.shape

        assert T <= self.config.context_length, "sequence longer than context_lenght"

        tok = self.token_emb(idx)
        pos = self.position_emb[:, :T, :]
        x = self.embedding_dropout(tok + pos)

        for block in self.blocks: 
            x = block(x)

        logits = self.lm_head(x)

        loss = None 
        if targets is not None: 
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1), targets.view(-1))
            )
        return logits, loss

    @torch.no_grad()

    def generate(self, idx, max_new_tokens, temperature = 1.0, top_k = None):
        # Autoregressive

        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.config.context_length:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature

            if top_k is not None: 
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = float("-inf")

            probs = F.softmax(logits, dim = -1)
            next_id = torch.multinomial(probs, num_samples = 1)
            idx = torch.cat((idx, next_id), dim = 1)

        return idx

    def num_params(self, non_embedding = False):
        n = sum(p.nuumel() for p in self.parameters())
        if non_embedding:
            n -= self.position_emb.numel()
        return n 


if __name__ == "__main__":
    torch.manual_seed(42)
    config = GPT1Config()
    model = GPT1(config)


    print(f"Total parameters: {model.num_params():,}")

    B, T = 2, 16 
    dummy_idx = torch.randint(0, config.vocab_size, (B, T))
    dummy_targets = torch.randint(0, config.vocab_size, (B, T))

    logits, loss = model(dummy_idx, dummy_targets)
    print("logits shape:", tuple(logits.shape))
    print("loss:", loss.item())

    generated = model.generate(dummy_idx, max_new_tokens= 5, top_k= 10)
    print("generated shape:", tuple(generated.shape))

In [ ]:
text = (
    "the quick brown fox jumps over the lazy dog. "
    "she sells seashells by the seashore. "
    "to be or not to be, that is the question. "
) * 50

chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s):
    return [stoi[c] for c in s]

def decode(ids):
    return "".join(itos[i] for i in ids)

data = torch.tensor(encode(text), dtype = torch.long)

config = GPT1Config(
    vocab_size = vocab_size, 
    context_length = 64,
    hidden_size = 128,
    n_layer = 4,
    num_head = 4, 
    intermediate_step = 512, 
    dropout = 0.1
)

model = GPT1(config)
print(f"Model parameters: {model.num_params():,}")

block_size = config.context_length
batch_size = 32

def get_batch():
    ix = torch.randint(0, len(data) - block_size -1, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[ i + 1:i + block_size + 1]] for i in ix)

optimizer = torch.optim.AdamW(model.parameters(), lr = 3e-4)

model.train()

for step in range(300):
    xb, yb = get_batch()
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none = True)
    loss.backward()
    optimizer.step()
    if step % 50 == 0 or step == 299:
        print(f"Step {step:4d} | Loss {loss.item():.4f}")


model.eval()
context = torch.tensor([encode("the ")], dtype = torch.long)
out = model.generate(context, max_new_tokens= 80, temperature= 0.8, top_k= 10)
print("\nSample generations:")
print(decode(out[0].tolist()))